# ניתוח רשת שיתופי הפעולה בקולנוע הישראלי
## ניתוח גרף חברתי באמצעות שיטות למידת מכונה

**תקציר:** מחקר זה בוחן את רשת שיתופי הפעולה בין 1,399 שחקנים בקולנוע הישראלי לאורך 107 שנים (1918-2025). המחקר כולל ניתוח מבני הרשת, זיהוי קהילות, חישוב מדדי מרכזיות, ומודל חיזוי לשיתופי פעולה עתידיים.

---

## 1. הגדרות והכנת הסביבה

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from bidi.algorithm import get_display

sys.path.append('../src')
from graph_build import load_cast_edges, build_full_graph, _actor_id

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 12

def he(s):
    """Fix Hebrew text for matplotlib"""
    return "\n".join(get_display(line) for line in str(s).split("\n"))

print("✅ כל הספריות נטענו בהצלחה!")

In [ ]:
# Load data
cast_df = load_cast_edges('../data/processed/cast_edges.csv')
cast_df['actor_id'] = [
    _actor_id(slug, name) 
    for slug, name in zip(cast_df['actor_slug'], cast_df['actor_name'])
]

# Build full graph
G = build_full_graph(cast_df)

print(f"✅ הגרף נטען: {G.number_of_nodes():,} שחקנים, {G.number_of_edges():,} קשרים")

---
## 2. מאפיינים בסיסיים של הרשת

### מדדים כמותיים עיקריים

In [ ]:
# Calculate key statistics
n_actors = G.number_of_nodes()
n_edges = G.number_of_edges()
density = nx.density(G)
n_movies = cast_df['movie_slug'].nunique()
avg_degree = sum(dict(G.degree()).values()) / n_actors

# Get giant component
giant = max(nx.connected_components(G), key=len)
giant_size = len(giant)
giant_pct = 100 * giant_size / n_actors

# Create impressive statistics display
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(he('סטטיסטיקות מרכזיות - הגרף במספרים'), fontsize=20, fontweight='bold', y=0.98)

stats = [
    (n_actors, he('שחקנים'), '#3b6ea5'),
    (n_edges, he('שיתופי פעולה'), '#e8743b'),
    (n_movies, he('סרטים'), '#19a974'),
    (f"{avg_degree:.1f}", he('ממוצע שותפים'), '#a463f2'),
    (f"{giant_pct:.0f}%", he('ברכיב הענק'), '#f7b731'),
    (f"{density*10000:.2f}", he('צפיפות (×10⁻⁴)'), '#eb3b5a')
]

for ax, (value, label, color) in zip(axes.flat, stats):
    ax.text(0.5, 0.6, f"{value:,}" if isinstance(value, int) else value, 
            ha='center', va='center', fontsize=48, fontweight='bold', color=color)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=18, color='#333')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, 
                               fill=False, edgecolor=color, linewidth=3, alpha=0.7))

plt.tight_layout()
plt.savefig('../figures/presentation_key_stats.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ סטטיסטיקות מרכזיות הוצגו!")

---
## 3. ויזואליזציה ומבנה קהילתי

### זיהוי קהילות באמצעות אלגוריתם Louvain

In [ ]:
from communities import run_algorithms

# Run community detection
print("מזהה קהילות...")
results = run_algorithms(G, seed=42)
louvain_communities = results['louvain']

# Create node-to-community mapping
node_to_comm = {}
for idx, comm in enumerate(louvain_communities):
    for node in comm:
        node_to_comm[node] = idx

print(f"✅ זוהו {len(louvain_communities)} קהילות")

In [ ]:
# Create beautiful graph visualization
print("יוצר ויזואליזציה מרשימה של הגרף...")

# Get giant component for visualization
giant_nodes = max(nx.connected_components(G), key=len)
G_giant = G.subgraph(giant_nodes).copy()

# Sample for visualization (top degree nodes)
degrees = dict(G_giant.degree())
top_nodes = sorted(degrees, key=degrees.get, reverse=True)[:300]
G_vis = G_giant.subgraph(top_nodes).copy()

# Create layout
print("מחשב מיקום צמתים...")
pos = nx.spring_layout(G_vis, seed=42, k=0.3, iterations=50)


# Find top 20 largest communities
comm_sizes = [(idx, len(comm)) for idx, comm in enumerate(louvain_communities)]
comm_sizes.sort(key=lambda x: x[1], reverse=True)
top_20_comms = {comm_id for comm_id, _ in comm_sizes[:20]}

print(f"מציג את 20 הקהילות הגדולות ביותר (מתוך {len(louvain_communities)} קהילות)")

# Assign colors: top 20 communities get distinct colors, others get gray
colors = []
for node in G_vis.nodes():
    comm_id = node_to_comm.get(node, -1)
    if comm_id in top_20_comms:
        # Map to color index (0-19)
        colors.append(list(top_20_comms).index(comm_id))
    else:
        # Gray for small communities
        colors.append(-1)

node_sizes = [degrees[node] * 30 for node in G_vis.nodes()]

# Plot
fig, ax = plt.subplots(figsize=(20, 20))
nx.draw_networkx_edges(G_vis, pos, ax=ax, alpha=0.15, width=0.5, edge_color='gray')

# Draw nodes in two groups: colored (top 20 communities) and gray (others)
colored_nodes = [n for n, c in zip(G_vis.nodes(), colors) if c >= 0]
gray_nodes = [n for n, c in zip(G_vis.nodes(), colors) if c < 0]
colored_colors = [c for c in colors if c >= 0]
colored_sizes = [degrees[n] * 30 for n in colored_nodes]
gray_sizes = [degrees[n] * 30 for n in gray_nodes]

# Draw gray nodes first (background)
if gray_nodes:
    nx.draw_networkx_nodes(G_vis.subgraph(gray_nodes), pos, ax=ax,
                          nodelist=gray_nodes,
                          node_size=gray_sizes,
                          node_color='#CCCCCC',
                          alpha=0.4,
                          linewidths=0.5,
                          edgecolors='white')

# Draw colored nodes (foreground)
if colored_nodes:
    nx.draw_networkx_nodes(G_vis.subgraph(colored_nodes), pos, ax=ax,
                          nodelist=colored_nodes,
                          node_size=colored_sizes,
                          node_color=colored_colors,
                          cmap=plt.cm.tab20,
                          alpha=0.85,
                          linewidths=1,
                          edgecolors='white')

# Add labels for top nodes
top_20 = sorted(degrees, key=degrees.get, reverse=True)[:20]
labels = {n: he(G.nodes[n].get('display_name', n).replace('_', ' ')) 
          for n in top_20 if n in G_vis.nodes()}
nx.draw_networkx_labels(G_vis, pos, labels, ax=ax, font_size=10, font_weight='bold')

ax.set_title(he(f'רשת שיתופי הפעולה - 20 הקהילות הגדולות (מתוך {len(louvain_communities)})'), 
             fontsize=24, fontweight='bold', pad=20)
ax.text(0.5, -0.05, he(f'מוצגים 300 השחקנים המרכזיים | צבעוניים: 20 קהילות גדולות | אפור: קהילות קטנות'), 
        ha='center', fontsize=14, transform=ax.transAxes)
ax.axis('off')

plt.tight_layout()
plt.savefig('../figures/presentation_network_communities.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n✅ ויזואליזציית הגרף הושלמה!")

---
## 4. ניתוח מרכזיות

### חמישה-עשר השחקנים בעלי Degree Centrality הגבוה ביותר

In [ ]:
# Calculate centralities
degree_cent = nx.degree_centrality(G)
top_15 = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:15]

# Prepare data
names = [he(G.nodes[actor].get('display_name', actor).replace('_', ' ')) for actor, _ in top_15]
values = [G.degree(actor) for actor, _ in top_15]

# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(14, 10))
bars = ax.barh(range(len(names)), values, color=plt.cm.viridis(np.linspace(0.3, 0.9, len(names))))

ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=13)
ax.invert_yaxis()
ax.set_xlabel(he('מספר שיתופי פעולה'), fontsize=14, fontweight='bold')
ax.set_title(he('15 השחקנים המרכזיים ביותר בקולנוע הישראלי'), 
             fontsize=18, fontweight='bold', pad=20)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, values)):
    ax.text(val + 1, i, f'{val}', va='center', fontsize=11, fontweight='bold')

ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/presentation_top_actors.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ השחקנים המרכזיים הוצגו!")

---
## 5. ניתוח טמפורלי

### השוואת מאפייני הרשת בשלוש תקופות היסטוריות

In [ ]:
from link_prediction import build_graph_for_years

# Build graphs for three periods
periods = [
    (1918, 1970, 'A', he('תקופה א: 1918-1970\n(הקולנוע המוקדם)')),
    (1971, 1990, 'B', he('תקופה ב: 1971-1990\n(תקופת המעבר)')),
    (1991, 2025, 'C', he('תקופה ג: 1991-2025\n(הקולנוע המודרני)'))
]

period_stats = []
for y_min, y_max, label, name in periods:
    G_period = build_graph_for_years(cast_df, y_min, y_max)
    period_stats.append({
        'name': name,
        'actors': G_period.number_of_nodes(),
        'collaborations': G_period.number_of_edges(),
        'density': nx.density(G_period) * 1000,
        'avg_degree': sum(dict(G_period.degree()).values()) / max(G_period.number_of_nodes(), 1)
    })

# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(he('התפתחות הגרף לאורך תקופות'), fontsize=20, fontweight='bold')

metrics = ['actors', 'collaborations', 'avg_degree', 'density']
titles = [he('מספר שחקנים'), he('מספר שיתופי פעולה'), he('ממוצע שותפים'), he('צפיפות (×10⁻³)')]
colors = ['#3b6ea5', '#e8743b', '#19a974']

for ax, metric, title in zip(axes.flat, metrics, titles):
    values = [s[metric] for s in period_stats]
    bars = ax.bar(range(3), values, color=colors, alpha=0.8, edgecolor='white', linewidth=2)
    
    ax.set_xticks(range(3))
    ax.set_xticklabels([s['name'] for s in period_stats], fontsize=10)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:,.0f}' if val >= 10 else f'{val:.1f}',
                ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/presentation_temporal_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ התפתחות לאורך זמן הוצגה!")

---
## 6. מודל חיזוי לשיתופי פעולה

### ביצועי המודל על מערכות בחן שונות

**תקופת אימון:** 1990-2020  
**Test Set (2016-2020):** AUC = 73.7%  
**Holdout Evaluation (2021-2025):** AUC = 65.7%

In [ ]:
# Load prediction results (separate files for hits and misses)
try:
    hits_df = pd.read_csv('../data/processed/single_horizon_true_positives_detailed.csv')
    misses_df = pd.read_csv('../data/processed/single_horizon_false_positives_detailed.csv')
    
    # Rename columns to match expected format
    if 'probability' in hits_df.columns:
        hits_df = hits_df.rename(columns={'probability': 'score'})
    if 'probability' in misses_df.columns:
        misses_df = misses_df.rename(columns={'probability': 'score'})
    
    hits_df['actually_connected'] = 1
    misses_df['actually_connected'] = 0
    
    # Combine for overall statistics
    results_df = pd.concat([hits_df, misses_df], ignore_index=True)
    
    tp = len(hits_df)
    fp = len(misses_df)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    print(f"✅ טעינת נתונים: {tp} הצלחות, {fp} כשלונות")
except FileNotFoundError as e:
    print(f"⚠️ שגיאה בטעינת קבצים: {e}")
    # Fallback to original file if needed
    results_df = pd.read_csv('../data/processed/future_predictions_2021_2025_single_horizon.csv')
    hits_df = results_df[results_df['actually_connected'] == 1]
    misses_df = results_df[results_df['actually_connected'] == 0]
    tp = len(hits_df)
    fp = len(misses_df)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

# Model performance scores from notebook 06b
test_auc = 0.737      # AUC on Test Set (2016-2020) - validation performance
holdout_auc = 0.657   # AUC on Holdout (2021-2025) - real-world evaluation

# Create results visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(he('תוצאות מודל הניבוי - ביצועים והצלחות'), fontsize=18, fontweight='bold')

# Chart 1: AUC scores (Test vs Holdout)
ax = axes[0]
auc_labels = [he('Test Set\n(2016-2020)'), he('Holdout\n(2021-2025)')]
auc_values = [test_auc * 100, holdout_auc * 100]
colors_auc = ['#3b6ea5', '#19a974']
bars = ax.bar([0, 1], auc_values, color=colors_auc, alpha=0.85, width=0.6, edgecolor='white', linewidth=2)
ax.set_ylim(0, 100)
ax.set_xlim(-0.5, 1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(auc_labels, fontsize=11)
ax.set_ylabel(he('AUC (%)'), fontsize=12)
ax.set_title(he('ביצועי המודל\nAUC Score'), fontsize=14, fontweight='bold')
for i, (bar, val) in enumerate(zip(bars, auc_values)):
    ax.text(i, val + 2, f'{val:.1f}%', ha='center', fontsize=16, fontweight='bold')
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.text(0.5, 52, he('Random (50%)'), ha='center', fontsize=9, color='gray')
ax.grid(axis='y', alpha=0.3)

# Chart 2: Hits breakdown (2021-2025 Holdout)
ax = axes[1]
categories = [he('ניבויים נכונים'), he('ניבויים שגויים')]
values = [tp, fp]
colors_pie = ['#19a974', '#e8743b']
wedges, texts, autotexts = ax.pie(values, labels=categories, colors=colors_pie, autopct='%1.0f%%',
                                    startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
ax.set_title(he(f'התפלגות התוצאות\n(הערכה על 2021-2025)'), fontsize=14, fontweight='bold')

# Chart 3: Score distribution
ax = axes[2]
connected = hits_df['score'] if len(hits_df) > 0 else []
not_connected = misses_df['score'] if len(misses_df) > 0 else []
if len(connected) > 0 and len(not_connected) > 0:
    ax.hist([connected, not_connected], bins=20, label=[he('התממשו'), he('לא התממשו')],
            color=['#19a974', '#e8743b'], alpha=0.7, edgecolor='white')
ax.set_xlabel(he('ציון חיזוי'), fontsize=12)
ax.set_ylabel(he('תדירות'), fontsize=12)
ax.set_title(he('התפלגות ציוני החיזוי'), fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/presentation_model_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 ביצועי המודל:")
print(f"   Test Set (2016-2020):    AUC = {test_auc:.3f} (73.7%)")
print(f"   Holdout (2021-2025):     AUC = {holdout_auc:.3f} (65.7%)")
print(f"\n✅ הערכה על 2021-2025: {tp} ניבויים נכונים מתוך {tp+fp} ({precision*100:.1f}% precision)")

In [ ]:
# Show successful predictions (hits)
if len(hits_df) > 0:
    top_hits = hits_df.sort_values('score', ascending=False).head(10)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    names = [he(f"{row['u'].replace('_', ' ')} ו{row['v'].replace('_', ' ')}") 
             for _, row in top_hits.iterrows()]
    scores = top_hits['score'].values
    
    bars = ax.barh(range(len(names)), scores, color='#19a974', alpha=0.8, edgecolor='white', linewidth=2)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=12)
    ax.invert_yaxis()
    ax.set_xlabel(he('ציון חיזוי'), fontsize=13)
    ax.set_title(he('10 הניבויים המוצלחים ביותר - שיתופי פעולה שהתממשו!'), 
                 fontsize=16, fontweight='bold', pad=15)
    
    for i, (bar, score) in enumerate(zip(bars, scores)):
        ax.text(score + 0.01, i, f'{score:.3f}', va='center', fontsize=11, fontweight='bold')
    
    ax.set_xlim(0, 1.05)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../figures/presentation_successful_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✅ הצלחות המודל הוצגו: {len(top_hits)} ניבויים מוצלחים!")
else:
    print("⚠️ אין ניבויים מוצלחים להצגה")

In [ ]:
# Show failed predictions (interesting misses)
if len(misses_df) > 0:
    top_misses = misses_df.sort_values('score', ascending=False).head(10)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    names = [he(f"{row['u'].replace('_', ' ')} ו{row['v'].replace('_', ' ')}") 
             for _, row in top_misses.iterrows()]
    scores = top_misses['score'].values
    
    bars = ax.barh(range(len(names)), scores, color='#e8743b', alpha=0.8, edgecolor='white', linewidth=2)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=12)
    ax.invert_yaxis()
    ax.set_xlabel(he('ציון חיזוי'), fontsize=13)
    ax.set_title(he('כשלונות מעניינים - ציפינו לשיתופי פעולה שלא התממשו'), 
                 fontsize=16, fontweight='bold', pad=15)
    
    for i, (bar, score) in enumerate(zip(bars, scores)):
        ax.text(score + 0.01, i, f'{score:.3f}', va='center', fontsize=11, fontweight='bold')
    
    ax.set_xlim(0, 1.05)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../figures/presentation_failed_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✅ כשלונות המודל הוצגו: {len(top_misses)} זוגות עם פוטנציאל גבוה שטרם שיתפו פעולה!")
else:
    print("⚠️ אין כשלונות להצגה")

---
## 7. מסקנות ותובנות

### ממצאים עיקריים מהניתוח

In [ ]:
# Create insights summary
fig = plt.figure(figsize=(16, 10))
fig.suptitle(he('תובנות מרכזיות מהמחקר'), fontsize=22, fontweight='bold', y=0.96)

# Use simple numbers/letters that matplotlib can display reliably
symbols = ["1", "2", "3", "4", "5", "6"]

insights = [
    (symbols[0], he("רשת מקושרת"), 
     he(f"{giant_pct:.0f}% מהשחקנים ברכיב ענק אחד\nמעיד על תעשייה מקושרת היטב")),
    
    (symbols[1], he("צמיחה מרשימה"), 
     he(f"תקופה ג׳ (1991-2025):\n{period_stats[2]['actors']:,} שחקנים\nפי-{period_stats[2]['actors']/period_stats[0]['actors']:.1f} מתקופה א׳")),
    
    (symbols[2], he("קהילות מובחנות"), 
     he(f"{len(louvain_communities)} קהילות זוהו\nמעידות על סגנונות ז׳אנרים שונים")),
    
    (symbols[3], he("שחקנים מרכזיים"), 
     he(f"השחקן המרכזי ביותר:\n{top_15[0][0].replace('_', ' ')}\n({values[0]:,} שיתופי פעולה)")),
    
    (symbols[4], he("מודל ניבוי מוצלח"), 
     he(f"Test: 73.7% AUC\nHoldout: 65.7% AUC\n122 ניבויים נכונים!")),
    
    (symbols[5], he("פוטנציאל עתידי"), 
     he(f"זוהו {fp:,} זוגות עם פוטנציאל גבוה\nשטרם שיתפו פעולה"))
]

colors_box = ['#3b6ea5', '#e8743b', '#19a974', '#a463f2', '#f7b731', '#eb3b5a']

for idx, (symbol, title, text) in enumerate(insights):
    ax = fig.add_subplot(2, 3, idx+1)
    
    # Display symbol in the same color as the box border
    ax.text(0.5, 0.75, symbol, ha='center', va='center', 
            fontsize=90, fontweight='bold', color=colors_box[idx])
    ax.text(0.5, 0.50, title, ha='center', va='center', 
            fontsize=16, fontweight='bold', color='#2c3e50')
    ax.text(0.5, 0.20, text, ha='center', va='center', 
            fontsize=12, color='#34495e', multialignment='center')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    ax.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, 
                               fill=True, facecolor=colors_box[idx], 
                               alpha=0.1, edgecolor=colors_box[idx], linewidth=3))

plt.tight_layout()
plt.savefig('../figures/presentation_insights.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n✅ תובנות מרכזיות הוצגו!")

---
## סיכום

### קבצי ויזואליזציה שנוצרו:
1. `presentation_key_stats.png` - מאפיינים כמותיים בסיסיים
2. `presentation_network_communities.png` - מבנה הרשת והקהילות המרכזיות
3. `presentation_top_actors.png` - שחקנים בעלי מרכזיות גבוהה
4. `presentation_temporal_evolution.png` - התפתחות הרשת על פני שלוש תקופות
5. `presentation_model_results.png` - ביצועי מודל החיזוי
6. `presentation_successful_predictions.png` - דוגמאות לחיזויים מוצלחים
7. `presentation_failed_predictions.png` - ניתוח חיזויים שגויים
8. `presentation_insights.png` - ממצאים מרכזיים

### מבנה המחקר:
1. **ניתוח תיאורי:** סטטיסטיקות בסיסיות ומבנה הרשת
2. **ניתוח מבני:** קהילות ומדדי מרכזיות
3. **ניתוח דינמי:** התפתחות הרשת לאורך זמן
4. **מודל חיזוי:** למידת מכונה לניבוי קשרים עתידיים

---
**מיקום הקבצים:** `israeli_actors_graph/figures/`

### תוצאות עיקריות:
- **היקף הרשת:** 1,399 צמתים, 6,092 קשרים
- **מבנה קהילתי:** 68 קהילות זוהו (20 הגדולות מוצגות)
- **מאפייני צפיפות:** ירידה במדד הצפיפות עם צמיחת הרשת
- **ביצועי חיזוי:** 122 חיזויים נכונים מתוך 210 (58.1% precision)
- **דיוק מודל:** AUC 73.7% (Test Set), AUC 65.7% (Holdout)